# 01: OTF の 3 層構造とメタデータ解剖（Iceberg / Delta Lake）

## このノートブックで学習できること

- Landing の Day 1 データを Bronze 層の Apache Iceberg テーブルおよび Delta Lake テーブルとして書き込む手順
- Iceberg の `metadata.json`・Manifest List（`snap-*.avro`）・Manifest File（`*-m0.avro`）と、Delta Lake の `_delta_log/*.json` を Python で直接開き、内部構造をツリー表示して比較する方法
- カタログ層・メタデータ層・データ層（Parquet）の 3 層構造によって、ストレージの遅い `LIST` 操作を行わずに Min / Max / Null 数の統計情報だけで Data Skipping（3段階プルーニング）が働く仕組み

## 到達ゴール

- Iceberg と Delta Lake がどちらも Parquet の上にトランザクションログ（メタデータ層）を持つことで ACID と高速プルーニングを実現していることを実ファイルから説明できる
- `plan_files()` が Manifest File の統計情報を使って不要な Parquet ファイルを読み取り対象から除外できることを確認できる

## なぜこれが必要か（従来の Hive 形式データレイクとの比較）

従来の Hive 形式データレイクでは、クエリエンジンがまずストレージに対して `LIST` リクエストを発行して全フォルダ・全ファイルを列挙し、さらに各 Parquet ファイルのフッターを開いて統計情報を確認する必要がありました。ファイル数が数万〜数百万に増えると、`LIST` だけで数分かかることが課題でした。OTF はメタデータ層にファイル一覧と列ごとの統計情報をあらかじめ集約することで、`LIST` なしの高速な 3 段階プルーニングを実現します。


## 1. Iceberg と Delta Lake への書き込み

このセルでは、Day 1 の `pos_transactions.jsonl` を読み込み、`transaction_id` の前半（1〜150）と後半（151〜300）の 2 回に分けて Bronze 層の Iceberg テーブル（`bronze.pos_transactions_d1`）と Delta Lake テーブル（`storage/lakehouse-main/bronze/pos_transactions_delta`）へ書き込みます。2 回に分けて書き込むことで、複数の Parquet ファイルと統計情報の差を観察できるようにします。


In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.catalog_helper import (
    LANDING_DIR,
    LAKEHOUSE_DIR,
    get_duckdb_conn,
    get_iceberg_catalog,
    init_storage,
    register_iceberg_table,
)
import json
import fastavro
import polars as pl
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake
from pyiceberg.expressions import EqualTo

catalog = get_iceberg_catalog()
df_pos = pl.read_ndjson(LANDING_DIR / "day1" / "pos_transactions.jsonl").sort("transaction_id")

batch1 = df_pos.slice(0, 150).to_arrow()
batch2 = df_pos.slice(150, 150).to_arrow()

# Iceberg テーブル作成 & 2バッチ追記
if catalog.table_exists("bronze.pos_transactions_d1"):
    catalog.drop_table("bronze.pos_transactions_d1")
ice_tbl = catalog.create_table("bronze.pos_transactions_d1", schema=batch1.schema)
ice_tbl.append(batch1)
ice_tbl.append(batch2)

# Delta Lake テーブル書き込み & 2バッチ追記
delta_path = LAKEHOUSE_DIR / "bronze" / "pos_transactions_delta"
write_deltalake(str(delta_path), batch1, mode="overwrite")
write_deltalake(str(delta_path), batch2, mode="append")

print(f"Iceberg snapshots: {len(ice_tbl.snapshots())}")
print(f"Delta version    : {DeltaTable(str(delta_path)).version()}")


Iceberg では 2 回の `append` によりスナップショット数が `2`、Delta Lake ではバージョンが `1`（0 と 1 の 2 コミット）になりました。

次のセルでは、Iceberg が生成した `metadata.json` → Manifest List（`snap-*.avro`）→ Manifest File（`*-m0.avro`）→ Parquet の 3 層構造を直接ファイルから読み取り、ツリー形式で表示します。


In [ ]:
def read_avro(path: Path) -> list[dict]:
    with path.open("rb") as f:
        return list(fastavro.reader(f))

meta_path = Path(ice_tbl.metadata_location.replace("file://", ""))
meta = json.loads(meta_path.read_text(encoding="utf-8"))
cur_snap = next(s for s in meta["snapshots"] if s["snapshot-id"] == meta["current-snapshot-id"])
mlist_path = Path(cur_snap["manifest-list"].replace("file://", ""))

print(f"[Catalog] bronze.pos_transactions_d1 -> {meta_path.relative_to(ROOT_DIR)}")
print(f"  └─ [Table Metadata] current-snapshot-id = {cur_snap['snapshot-id']}")
print(f"       └─ [Manifest List] {mlist_path.name}")

for m_entry in read_avro(mlist_path):
    m_path = Path(m_entry["manifest_path"].replace("file://", ""))
    print(f"            ├─ [Manifest File] {m_path.name} (added_files={m_entry['added_files_count']})")
    for d_entry in read_avro(m_path):
        df_info = d_entry["data_file"]
        pq_name = Path(df_info["file_path"]).name
        print(f"            │    └─ [Parquet] {pq_name} (records={df_info['record_count']}, size={df_info['file_size_in_bytes']}B)")


出力されたツリーから、Catalog が `metadata.json` を指し、`metadata.json` が最新の Manifest List を指し、さらに各 Manifest File が個別の Parquet ファイル（各 150 レコード）を指す 3 層構造が確認できました。

次のセルでは、比較対象として Delta Lake の `_delta_log/` ディレクトリにある JSON トランザクションログを直接開き、書き込まれたエントリと `stats`（`minValues` / `maxValues` / `nullCount`）を表示します。


In [ ]:
delta_log_dir = delta_path / "_delta_log"
for log_file in sorted(delta_log_dir.glob("*.json")):
    print(f"=== {log_file.name} ===")
    for line in log_file.read_text(encoding="utf-8").splitlines():
        entry = json.loads(line)
        if "add" in entry:
            add = entry["add"]
            stats = json.loads(add["stats"])
            print(f"  [add] {add['path']} | rows={stats['numRecords']} | min={stats['minValues']['transaction_id']} | max={stats['maxValues']['transaction_id']}")


Delta Lake でも `_delta_log/00000000000000000000.json` と `00001.json` の中に、追加された Parquet ファイル名と `minValues`・`maxValues` が JSON として記録されていることが分かります。

次のセルでは、Iceberg の Manifest File にある `lower_bounds` / `upper_bounds` と `null_value_counts` を取り出して一覧表示し、さらに `transaction_id = 'TX-000250'` の検索時に Data Skipping が働いて 2 ファイルのうち 1 ファイルだけしかスキャンされないことを確認します。


In [ ]:
def to_map(obj) -> dict:
    if isinstance(obj, dict):
        return obj
    return {item["key"]: item["value"] for item in obj}

manifest_rows = []
for m_entry in read_avro(mlist_path):
    m_path = Path(m_entry["manifest_path"].replace("file://", ""))
    for d_entry in read_avro(m_path):
        df_info = d_entry["data_file"]
        lbounds = to_map(df_info["lower_bounds"])
        ubounds = to_map(df_info["upper_bounds"])
        ncounts = to_map(df_info["null_value_counts"])
        manifest_rows.append({
            "parquet": Path(df_info["file_path"]).name,
            "records": df_info["record_count"],
            "tx_min": lbounds[1].decode("utf-8"),
            "tx_max": ubounds[1].decode("utf-8"),
            "tx_nulls": ncounts[1],
        })

display_df = pl.DataFrame(manifest_rows).sort("tx_min")
print(display_df)

all_tasks = list(ice_tbl.scan().plan_files())
skip_tasks = list(ice_tbl.scan(row_filter=EqualTo("transaction_id", "TX-000250")).plan_files())
print(f"全件スキャン対象ファイル数       : {len(all_tasks)}")
print(f"TX-000250 絞り込み後の対象ファイル数: {len(skip_tasks)} -> {Path(skip_tasks[0].file.file_path).name}")


`TX-000001 .. TX-000150` を持つ 1 つ目の Parquet ファイルは Manifest File の統計情報だけでスキップされ、`TX-000151 .. TX-000300` を持つ 1 ファイルだけが読み取り対象に選ばれました。ストレージへの `LIST` 操作も、不要な Parquet ファイルのオープンも発生していません。
